# GLM-OCR (+ HunyuanOCR) sur GPU Kaggle -- manuscrit VN/EN

Meme jeux que les deux autres notebooks (`handwriting_vi`, `handwriting_en`,
30 pages chacun), pour GLM-OCR et HunyuanOCR-1.5 -- les deux modeles qui
partagent le meme `venv_hunyuan` en local (transformers recent, incompatible
avec la pile Paddle et avec Chandra). Notebook separe pour la meme raison
que Chandra l'est de Paddle.

## Pourquoi ces deux modeles ici

**GLM-OCR** (0,9 B, MIT) est le plus interessant des deux pour ce
benchmark : le plus petit, le plus rapide (0,67 img/s annonce), et le
premier sur les benchmarks de tableaux -- mais le vietnamien n'est pas dans
ses langues officielles (chinois, anglais, francais, espagnol, russe,
allemand, japonais, coreen), et ce sont precisement les diacritiques
vietnamiennes qui ont fait decrocher PP-StructureV3 par le passe. Voir
`docs/glm_ocr.md` pour le detail du pari.

**HunyuanOCR-1.5** (1 B) tourne dans la meme cellule d'installation, donc
autant le mesurer aussi -- **mais sa licence communautaire Tencent exclut
l'UE, le Royaume-Uni et la Coree du Sud**, et Kaggle ne permet pas de
choisir la region de la machine. Sa cellule est **commentee par defaut** :
a decommenter seulement apres avoir tranche la question de juridiction avec
Thanh-Ha DO. GLM (MIT) n'a aucune de ces contraintes et peut tourner sans
reserve.

## Avant de lancer

1. **Settings -> Accelerator -> GPU T4 x2**
2. **Settings -> Internet -> On**
3. **Add Input -> Datasets** : `images_handwriting_vi.zip` et/ou
   `images_handwriting_en.zip`
4. Depot GitHub a jour

## 1. Verifier le GPU

In [ ]:
!nvidia-smi
import sys
print(sys.version)

## 2. Recuperer le code

In [ ]:
import os, subprocess

os.environ["GIT_TERMINAL_PROMPT"] = "0"

OWNER, REPO, SECRET = "Halfaa", "PTITOcr", "GITHUB_TOKEN"

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(SECRET)
    print(f"Secret '{SECRET}' trouve ({len(token)} caracteres)")
except ImportError:
    print("kaggle_secrets indisponible : tu n'es pas sur Kaggle ?")
except Exception as e:
    print(f"Secret '{SECRET}' illisible -> {type(e).__name__}: {e}")
    print("A verifier : Add-ons -> Secrets rattache a CE notebook,")
    print(f"label exactement '{SECRET}'.")

url = (f"https://{token}@github.com/{OWNER}/{REPO}.git" if token
       else f"https://github.com/{OWNER}/{REPO}.git")

DEPOT = f"/kaggle/working/{REPO}"
%cd /kaggle/working
!rm -rf {REPO}
r = subprocess.run(["git", "clone", "--depth", "1", url, REPO],
                   capture_output=True, text=True, timeout=180)
if r.returncode != 0:
    msg = r.stderr.replace(token, "***") if token else r.stderr
    raise SystemExit("Clone impossible.\n" + msg)

%cd {DEPOT}
!ls

## 3. Installer les dependances

L'architecture `glm_ocr` est trop recente pour un `transformers` publie :
il faut la version de developpement depuis GitHub. HunyuanOCR partage la
meme exigence (>= 5.13).

In [ ]:
import subprocess, sys, importlib.metadata as md

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", *args],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("\n".join(r.stdout.splitlines()[-5:]))
        print("\n".join(r.stderr.splitlines()[-10:]))
    return r.returncode == 0

torch_avant = md.version("torch")
print(f"torch preinstalle par Kaggle : {torch_avant}")

assert pip("-q", "git+https://github.com/huggingface/transformers.git"), \
    "installation de transformers (dev) echouee"
assert pip("-q", "accelerate", "pillow", "torchvision"), "installation echouee"

torch_apres = md.version("torch")
if torch_apres != torch_avant:
    print(f"\nATTENTION : torch est passe de {torch_avant} a {torch_apres}.")
    print(f"    !pip install -q --force-reinstall torch=={torch_avant}")

for pkg in ("transformers", "torch", "torchvision", "accelerate", "pillow"):
    try:
        print(f"{pkg:<16}{md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:<16}absent")

In [ ]:
import subprocess, sys

check = subprocess.run([sys.executable, "-c", """
import torch, transformers
print('torch       ', torch.__version__, '| cuda:', torch.cuda.is_available())
print('transformers', transformers.__version__)
assert torch.cuda.is_available(), 'torch ne voit pas le GPU'
maj, mino = (int(x) for x in transformers.__version__.split('.')[:2])
assert (maj, mino) >= (5, 13), f'transformers {transformers.__version__} trop ancien (>= 5.13 requis)'
"""], capture_output=True, text=True)

print(check.stdout)
if check.returncode != 0:
    print(check.stderr[-1500:])
    raise SystemExit("Environnement inutilisable -- ne lance pas le benchmark.")
print("Environnement OK.")

## 4. Reperer les jeux d'images

Fichiers `specimen_NNN.png`, 30 par langue. Un jeu absent ne bloque pas.

In [ ]:
from pathlib import Path

MOTIFS = {
    "handwriting_vi": ["handwriting_vi", "handwriting-vi", "images_handwriting_vi"],
    "handwriting_en": ["handwriting_en", "handwriting-en", "images_handwriting_en"],
}
PAGES_ATTENDUES = 30

dossiers_avec_images = {}
for png in Path("/kaggle/input").rglob("specimen_*.png"):
    dossiers_avec_images[png.parent] = dossiers_avec_images.get(png.parent, 0) + 1

assert dossiers_avec_images, "Aucune image : as-tu ajoute un dataset via Add Input ?"

print("Dossiers trouves :")
for dossier, n in sorted(dossiers_avec_images.items(), key=lambda kv: -kv[1]):
    print(f"  {n:>4} images   {dossier}")

def normalise(chemin):
    return str(chemin).lower().replace("-", "_")

JEUX, manquants, pris = {}, [], set()
for nom, ecritures in MOTIFS.items():
    retenus = []
    for ecriture in ecritures:
        retenus = [d for d in dossiers_avec_images
                   if ecriture.replace("-", "_") in normalise(d) and d not in pris]
        if retenus:
            break
    if not retenus:
        manquants.append(nom)
        continue
    if len(retenus) > 1:
        raise SystemExit(f"'{nom}' correspond a {len(retenus)} dossiers, detache les datasets en trop.")
    JEUX[nom] = retenus[0]
    pris.add(retenus[0])

print("\nJeux retenus :")
for nom, chemin in JEUX.items():
    n = len(list(chemin.glob("specimen_*.png")))
    marque = "" if n == PAGES_ATTENDUES else f"   <-- {PAGES_ATTENDUES} attendus !"
    print(f"  {nom:<18}{n:>4} pages   {chemin.name}{marque}")

if manquants:
    print("\nJeux ABSENTS, leurs branches seront sautees :", ", ".join(manquants))

assert JEUX, "Aucun jeu reconnu -- verifie les noms de dataset attaches." 

## 5. Lancer GLM-OCR

MIT, sans reserve geographique. Smoke test implicite : la premiere page de
chaque jeu dira vite si les diacritiques vietnamiennes passent ou non.

In [ ]:
import subprocess, sys

runs = {}
for nom, chemin in JEUX.items():
    run = f"{nom}_glm"
    runs[nom] = run
    print(f"\n{'=' * 60}\n{nom} -> {run}\n{'=' * 60}", flush=True)
    subprocess.run(
        [sys.executable, "-u", "scripts/bench_deformed.py", "--model", "glm",
         "--images-dir", str(chemin), "--output", run],
        cwd=DEPOT, check=True)

print("\nRuns produits :", list(runs.values()))

## 6. Rapatrier (GLM)

In [ ]:
import shutil
from pathlib import Path

base = Path(DEPOT) / "results"
zips_produits = []
for nom, run in runs.items():
    src = base / run
    assert src.exists(), f"{src} absent : le benchmark a-t-il tourne ?"
    out = shutil.make_archive(f"/kaggle/working/{run}", "zip", base, run)
    zips_produits.append(run)
    print(f"{Path(out).name:<40}{Path(out).stat().st_size / 1e6:>7.1f} Mo")

print("\nContenu de /kaggle/working :")
for f in sorted(Path("/kaggle/working").iterdir()):
    print(" ", f.name)

---
## 7. Variante HunyuanOCR -- a decommenter apres accord sur la licence

**Attention licence.** La licence communautaire Tencent exclut l'Union
europeenne, le Royaume-Uni et la Coree du Sud de son Territoire, et Kaggle
ne permet pas de choisir la region de la machine. GLM-OCR (MIT) n'a aucune
de ces contraintes. A trancher avec Thanh-Ha DO avant de lancer Hunyuan
ici.

In [ ]:
# runs_hunyuan = {}
# for nom, chemin in JEUX.items():
#     run = f"{nom}_hunyuan"
#     runs_hunyuan[nom] = run
#     subprocess.run(
#         [sys.executable, "-u", "scripts/bench_deformed.py", "--model", "hunyuan",
#          "--images-dir", str(chemin), "--output", run],
#         cwd=DEPOT, check=True)
#
# for nom, run in runs_hunyuan.items():
#     src = base / run
#     out = shutil.make_archive(f"/kaggle/working/{run}", "zip", base, run)
#     print(Path(out).name)
#
# shutil.rmtree(DEPOT, ignore_errors=True)

## Nettoyage

Le depot clone pese pour rien dans l'Output -- retire apres la derniere
cellule que tu as executee (GLM seul, ou GLM + Hunyuan).

In [ ]:
import shutil
shutil.rmtree(DEPOT, ignore_errors=True)
print("Depot retire.")

## 8. Ensuite, sur ton PC

```powershell
python src\eval_handwriting.py --run handwriting_vi_glm --verite-dir handwriting/generated --lang vi --detail
python src\eval_handwriting.py --run handwriting_en_glm --verite-dir handwriting_en/generated --lang en --detail
```

Si le pari GLM se confirme (diacritiques lues correctement), c'est le
candidat de deploiement le plus evident -- 0,9 B, MIT, tourne meme sur la
RTX 3050 Ti en local, sans GPU loue.